To predict the next tokens, we need to load both the tokenizer and the model. We'll use `AutoModelForCausalLM` for this purpose.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import torch.nn.functional as F

# Define the prompts as a list
prompt_texts = [
"One must have a mind of winter",
"To regard the frost and the boughs",
"Of the pine-trees crusted with snow;"
"And have been cold a long time",
"To behold the junipers shagged with ice,",
"The spruces rough in the distant glitter",
"Of the January sun; and not to think",
"Of any misery in the sound of the wind,",
"In the sound of a few leaves,",
"Which is the sound of the land",
"Full of the same wind",
"That is blowing in the same bare place",
"For the listener, who listens in the snow,",
"And, nothing himself, beholds",
"Nothing that is not there and the nothing that is.",
]

def predict_next_tokens(model_name, prompt):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name)

    # Encode the prompt
    input_ids = tokenizer.encode(prompt, return_tensors='pt')

    # Get model predictions (logits)
    with torch.no_grad():
        outputs = model(input_ids)
        logits = outputs.logits

    # Get the logits for the last token
    last_token_logits = logits[0, -1, :]

    # Convert logits to probabilities
    probabilities = F.softmax(last_token_logits, dim=-1)

    # Get the top 10 token probabilities and their indices
    top_k_probs, top_k_indices = torch.topk(probabilities, 10)

    print(f"\n--- Model: {model_name} ---")
    print(f"Prompt: '{prompt}'")
    print("Top 10 next tokens:")
    for i, (prob, idx) in enumerate(zip(top_k_probs, top_k_indices)):
        token = tokenizer.decode([idx])
        print(f"{i+1}. Token: '{token}', Probability: {prob.item()*100:.2f}%")

    # Get the 7th highest probability token
    seventh_token_idx = top_k_indices[6] # 6 because it's 0-indexed
    seventh_token_prob = top_k_probs[6]
    seventh_token = tokenizer.decode([seventh_token_idx]).strip() # Strip to remove potential leading spaces
    print(f"\n7th highest probability token: '{seventh_token}', Probability: {seventh_token_prob.item()*100:.2f}%")

    # Replace the last word of the prompt with the 7th closest token
    words = prompt.split()
    if words:
        new_prompt = " ".join(words[:-1]) + (" " if words[:-1] else "") + seventh_token
    else:
        new_prompt = seventh_token # Handle empty prompt case

    print(f"\nOriginal Prompt: '{prompt}'")
    print(f"Modified Prompt (with 7th token): '{new_prompt}'")

    return top_k_probs, top_k_indices, tokenizer, new_prompt

# Initialize a list to store all modified prompts
combined_modified_prompts = []

# Run for openai-community/gpt2 for each prompt
for prompt_text in prompt_texts:
    gpt2_probs, gpt2_indices, gpt2_tokenizer, modified_gpt2_prompt = predict_next_tokens("openai-community/gpt2", prompt_text)
    combined_modified_prompts.append(modified_gpt2_prompt)

print("\n--- Combined Modified Prompts ---")
for i, modified_prompt in enumerate(combined_modified_prompts):
    print(f"Prompt {i+1}: '{modified_prompt}'")


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]


--- Model: openai-community/gpt2 ---
Prompt: 'One must have a mind of winter'
Top 10 next tokens:
1. Token: '.', Probability: 20.07%
2. Token: ',', Probability: 15.17%
3. Token: ' to', Probability: 7.02%
4. Token: ' and', Probability: 6.42%
5. Token: ' for', Probability: 3.39%
6. Token: ' in', Probability: 2.66%
7. Token: ' that', Probability: 1.96%
8. Token: '."', Probability: 1.69%
9. Token: '!', Probability: 1.51%
10. Token: ' when', Probability: 1.48%
11. Token: ' or', Probability: 1.38%
12. Token: ' if', Probability: 1.34%
13. Token: ':', Probability: 1.32%
14. Token: '
', Probability: 1.22%
15. Token: ',"', Probability: 1.09%
16. Token: ';', Probability: 1.08%
17. Token: ' before', Probability: 0.81%
18. Token: 'y', Probability: 0.78%
19. Token: ' (', Probability: 0.64%
20. Token: ' at', Probability: 0.64%

7th highest probability token: 'at', Probability: 0.64%

Original Prompt: 'One must have a mind of winter'
Modified Prompt (with 7th token): 'One must have a mind of at'


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]


--- Model: openai-community/gpt2 ---
Prompt: 'To regard the frost and the boughs'
Top 10 next tokens:
1. Token: ' of', Probability: 25.18%
2. Token: ',', Probability: 19.92%
3. Token: ' as', Probability: 17.92%
4. Token: ' that', Probability: 3.24%
5. Token: ' in', Probability: 3.01%
6. Token: ' on', Probability: 2.77%
7. Token: ' and', Probability: 2.58%
8. Token: ' which', Probability: 1.65%
9. Token: ' from', Probability: 1.46%
10. Token: ' to', Probability: 0.73%
11. Token: ' for', Probability: 0.68%
12. Token: ' at', Probability: 0.57%
13. Token: ' (', Probability: 0.47%
14. Token: ' we', Probability: 0.46%
15. Token: ' with', Probability: 0.41%
16. Token: ' the', Probability: 0.40%
17. Token: ' I', Probability: 0.34%
18. Token: '.', Probability: 0.32%
19. Token: ' above', Probability: 0.32%
20. Token: ':', Probability: 0.29%

7th highest probability token: ':', Probability: 0.29%

Original Prompt: 'To regard the frost and the boughs'
Modified Prompt (with 7th token): 'To regard 

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

...


--- Model: openai-community/gpt2 ---
Prompt: 'For the listener, who listens in the snow,'
Top 10 next tokens:
1. Token: ' the', Probability: 10.17%
2. Token: ' you', Probability: 7.58%
3. Token: ' it', Probability: 4.53%
4. Token: ' he', Probability: 3.28%
5. Token: ' there', Probability: 3.12%
6. Token: ' I', Probability: 2.94%
7. Token: ' this', Probability: 2.68%
8. Token: ' we', Probability: 2.52%
9. Token: ' see', Probability: 2.18%
10. Token: ' and', Probability: 2.09%
11. Token: ' is', Probability: 1.85%
12. Token: ' a', Probability: 1.52%
13. Token: ' they', Probability: 1.31%
14. Token: ' who', Probability: 1.19%
15. Token: ' or', Probability: 1.09%
16. Token: ' what', Probability: 0.94%
17. Token: ' listen', Probability: 0.93%
18. Token: ' hear', Probability: 0.90%
19. Token: '
', Probability: 0.82%
20. Token: ' here', Probability: 0.75%

7th highest probability token: 'here', Probability: 0.75%

Original Prompt: 'For the listener, who listens in the snow,'
Modified Prompt (

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]


--- Model: openai-community/gpt2 ---
Prompt: 'And, nothing himself, beholds'
Top 10 next tokens:
1. Token: ' the', Probability: 19.69%
2. Token: ' more', Probability: 7.62%
3. Token: ' a', Probability: 5.22%
4. Token: ' this', Probability: 2.94%
5. Token: ' it', Probability: 2.08%
6. Token: ' that', Probability: 2.00%
7. Token: ' what', Probability: 1.89%
8. Token: ' his', Probability: 1.86%
9. Token: ',', Probability: 1.72%
10. Token: ' such', Probability: 1.55%
11. Token: ' so', Probability: 1.53%
12. Token: ' or', Probability: 1.52%
13. Token: ' us', Probability: 1.52%
14. Token: '.', Probability: 1.47%
15. Token: ' any', Probability: 1.36%
16. Token: ' me', Probability: 1.29%
17. Token: ' him', Probability: 1.19%
18. Token: ' in', Probability: 1.07%
19. Token: ' our', Probability: 0.99%
20. Token: ' better', Probability: 0.94%

7th highest probability token: 'better', Probability: 0.94%

Original Prompt: 'And, nothing himself, beholds'
Modified Prompt (with 7th token): 'And, nothi

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]


--- Model: openai-community/gpt2 ---
Prompt: 'Nothing that is not there and the nothing that is.'
Top 10 next tokens:
1. Token: '
', Probability: 18.55%
2. Token: ' And', Probability: 7.31%
3. Token: ' It', Probability: 5.46%
4. Token: ' I', Probability: 5.06%
5. Token: ' The', Probability: 4.81%
6. Token: ' That', Probability: 4.19%
7. Token: ' So', Probability: 2.78%
8. Token: ' You', Probability: 2.19%
9. Token: ' But', Probability: 2.12%
10. Token: ' There', Probability: 2.08%
11. Token: ' This', Probability: 2.04%
12. Token: ' We', Probability: 1.88%
13. Token: ' If', Probability: 1.79%
14. Token: '

', Probability: 1.30%
15. Token: ' He', Probability: 1.17%
16. Token: ' What', Probability: 1.15%
17. Token: ' In', Probability: 1.10%
18. Token: ' When', Probability: 0.93%
19. Token: ' They', Probability: 0.91%
20. Token: ' A', Probability: 0.76%

7th highest probability token: 'A', Probability: 0.76%

Original Prompt: 'Nothing that is not there and the nothing that is.'
Modified P

Original Poem:

One must have a mind of winter

To regard the frost and the boughs

Of the pine-trees crusted with snow;

And have been cold a long time

To behold the junipers shagged with ice,

The spruces rough in the distant glitter

Of the January sun; and not to think

Of any misery in the sound of the wind,

In the sound of a few leaves,

Which is the sound of the land

Full of the same wind

That is blowing in the same bare place

For the listener, who listens in the snow,

And, nothing himself, beholds

Nothing that is not there and the nothing that is.

N+7 Poem

One must have a mind of that

To regard the frost and the and

Of the pine-trees crusted with it;

And have been cold a long in

To behold the junipers shagged with as,

The spruces rough in the distant -

Of the January sun; and not to I

Of any misery in the sound of the that,

In the sound of a few coming,

Which is the sound of the in

Full of the same as

That is blowing in the same bare,

For the listener, who listens in the this,

And, nothing himself, what

Nothing that is not there and the nothing that So.

N+20 Poem

'One must have a mind of at'

'To regard the frost and the :'

'Of the pine-trees crusted with snow; And have been cold a long "'

'To behold the junipers shagged with was'

'The spruces rough in the distant like'

'Of the January sun; and not to to'

'Of any misery in the sound of the my'

'In the sound of a few two'

'Which is the sound of the shaking'

'Full of the same at'

'That is blowing in the same bare it'

'For the listener, who listens in the here'

'And, nothing himself, better'

'Nothing that is not there and the nothing that A'